In [0]:
from pyspark.sql.functions import col, expr, trim

# Read data
df = spark.read.table("mini_project_catalog.`01_bronze`.order")

# -----------------------------
# 1. Remove duplicates
# -----------------------------
df = df.dropDuplicates()

# -----------------------------
# 2. Replace invalid values ("-") with null
# -----------------------------
df = df.replace("-", None)

# -----------------------------
# 3. Trim string columns (clean spaces)
# -----------------------------
for c in df.columns:
    df = df.withColumn(c, trim(col(c)))

# -----------------------------
# 4. Convert timestamp columns (ISO format)
# -----------------------------
timestamp_cols = [
    "vehicle_in_datetime",
    "vehicle_out_datetime",
    "planned_work_start_datetime",
    "actual_work_start_datetime",
    "planned_completion_datetime",
    "actual_completion_datetime",
    "promised_delivery_datetime",
    "actual_delivery_datetime"
]

for c in timestamp_cols:
    if c in df.columns:
        df = df.withColumn(c, expr(f"to_timestamp({c})"))

# -----------------------------
# 5. Convert numeric columns
# -----------------------------
if "store_id" in df.columns:
    df = df.withColumn("store_id", col("store_id").cast("int"))

# -----------------------------
# 6. Filter important records
# -----------------------------
df = df.filter(col("order_id").isNotNull())

# -----------------------------
# 7. Standardize text columns
# -----------------------------
df = df.withColumn("order_status", col("order_status"))

# -----------------------------
# 8. Optional: Handle phone cleanup (basic)
# -----------------------------
df = df.withColumn("customer_phone", trim(col("customer_phone")))

# -----------------------------
# 9. Write to Silver
# -----------------------------
df.write.mode("overwrite").saveAsTable(
    "mini_project_catalog.02_silver.order"
)

In [0]:
%sql
SELECT * FROM mini_project_catalog.02_silver.order LIMIT 10;